In [ ]:
import rasterio
from rasterio.mask import mask
from shapely.geometry import box
import numpy as np
import pandas as pd
import geopandas as gpd

In [ ]:
# check if sao paolo exist in the geoboundries

# Path to the geojson file
boundary_file = "geoBoundaries-BRA-ADM1-all\geoBoundaries-BRA-ADM1.geojson"

# Load the GeoJSON file
brazil_boundaries = gpd.read_file(boundary_file)

city_name_column = "shapeName"

sao_paulo_boundary = brazil_boundaries[brazil_boundaries[city_name_column] == "Sao Paulo"]

# Check if the boundary is found
if not sao_paulo_boundary.empty:
    print("sao paulo boundary boundary found!")
else:
    raise ValueError("sao paulo boundary boundary not found. Check the city name or column.")

sao_paulo_boundary.to_file("sao_paulo_boundary.geojson", driver="GeoJSON")

# Print the coordinates of the boundary (or process further as needed)
print(sao_paulo_boundary)

sao paulo boundary boundary found!
    shapeName shapeISO                  shapeID shapeGroup shapeType  \
23  Sao Paulo    BR-SP  14911670B46470234103867        BRA      ADM1   

                                             geometry  
23  POLYGON ((-51.03033 -20.02491, -51.03137 -20.0...  


In [ ]:
# Filepaths

# raster_file = "SVDNB_npp_20200101-20200131_00N060W_vcmcfg_v10_c202002111500\SVDNB_npp_20200101-20200131_00N060W_vcmcfg_v10_c202002111500.avg_rade9h.tif"
# raster_file = "SVDNB_npp_20200201-20200229_00N060W_vcmcfg_v10_c202003021200\SVDNB_npp_20200201-20200229_00N060W_vcmcfg_v10_c202003021200.avg_rade9h.tif"
# raster_file = "SVDNB_npp_20200301-20200331_00N060W_vcmcfg_v10_c202007042300\SVDNB_npp_20200301-20200331_00N060W_vcmcfg_v10_c202007042300.avg_rade9h.tif"
# raster_file = "SVDNB_npp_20200401-20200430_00N060W_vcmcfg_v10_c202006121200\SVDNB_npp_20200401-20200430_00N060W_vcmcfg_v10_c202006121200.avg_rade9h.tif"
# raster_file = "SVDNB_npp_20200501-20200531_00N060W_vcmcfg_v10_c202006221000\SVDNB_npp_20200501-20200531_00N060W_vcmcfg_v10_c202006221000.avg_rade9h.tif"
# raster_file = "SVDNB_npp_20200601-20200630_00N060W_vcmcfg_v10_c202008012300\SVDNB_npp_20200601-20200630_00N060W_vcmcfg_v10_c202008012300.avg_rade9h.tif"
# raster_file = "SVDNB_npp_20200701-20200731_00N060W_vcmcfg_v10_c202008142300\SVDNB_npp_20200701-20200731_00N060W_vcmcfg_v10_c202008142300.avg_rade9h.tif"
# raster_file = "SVDNB_npp_20200801-20200831_00N060W_vcmcfg_v10_c202009111000\SVDNB_npp_20200801-20200831_00N060W_vcmcfg_v10_c202009111000.avg_rade9h.tif"
# raster_file = "SVDNB_npp_20200901-20200930_00N060W_vcmcfg_v10_c202010082300\SVDNB_npp_20200901-20200930_00N060W_vcmcfg_v10_c202010082300.avg_rade9h.tif"
# raster_file = "SVDNB_npp_20201001-20201031_00N060W_vcmcfg_v10_c202011050900\SVDNB_npp_20201001-20201031_00N060W_vcmcfg_v10_c202011050900.avg_rade9h.tif"
# raster_file = "SVDNB_npp_20201101-20201130_00N060W_vcmcfg_v10_c202012092300\SVDNB_npp_20201101-20201130_00N060W_vcmcfg_v10_c202012092300.avg_rade9h.tif"
raster_file = "SVDNB_npp_20201201-20201231_00N060W_vcmcfg_v10_c202101130900\SVDNB_npp_20201201-20201231_00N060W_vcmcfg_v10_c202101130900.avg_rade9h.tif"

sao_boundary_file = "sao_paulo_boundary.geojson"

# output_csv = "viirs_20_sao_1.csv"
# output_csv = "viirs_20_sao_2.csv"
# output_csv = "viirs_20_sao_3.csv"
# output_csv = "viirs_20_sao_4.csv"
# output_csv = "viirs_20_sao_5.csv"
# output_csv = "viirs_20_sao_6.csv"
# output_csv = "viirs_20_sao_7.csv"
# output_csv = "viirs_20_sao_8.csv"
# output_csv = "viirs_20_sao_9.csv"
# output_csv = "viirs_20_sao_10.csv"
# output_csv = "viirs_20_sao_11.csv"
output_csv ="viirs_20_sao_12.csv"

# Load the Brasília boundary
sao = gpd.read_file(sao_boundary_file)
sao = sao.to_crs(epsg=4326)

# Open the VIIRS raster file
with rasterio.open(raster_file) as src:
    raster_bounds = box(*src.bounds)
    print("Raster Bounds:", src.bounds)
    print("sao Bounds:", sao.total_bounds)

    if not raster_bounds.intersects(sao.unary_union):
        raise ValueError("sao's boundary does not overlap with the raster extent.")

    # Clip the VIIRS raster using Brasília boundary
    sao_geom_list = [feature["geometry"] for feature in sao.__geo_interface__["features"]]
    clipped_raster, clipped_transform = mask(src, sao_geom_list, crop=True)

# Extract Raster Values for Brasília
light_intensity = clipped_raster[0]
rows, cols = np.where(~np.isnan(light_intensity))
values = light_intensity[rows, cols]
x_coords, y_coords = rasterio.transform.xy(clipped_transform, rows, cols)

# Create a DataFrame
data = pd.DataFrame({
    'longitude': x_coords,
    'latitude': y_coords,
    'light_intensity': values
})

# Save the data to a CSV file
data.to_csv(output_csv, index=False)
print(f"Extracted data for sao saved to {output_csv}")


Raster Bounds: BoundingBox(left=-60.00208333335, bottom=-64.99791718665, right=59.99791762665001, top=0.00208333335)
sao Bounds: [-53.13444525 -25.34023719 -44.24107085 -19.80947155]


C:\Users\PC\AppData\Local\Temp\ipykernel_8836\501769821.py:23: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  if not raster_bounds.intersects(sao.unary_union):


Extracted data for sao saved to D:\NU\semester_5\data_analysis\project\edited_project\viirs_sao_3_dataset.csv


In [ ]:
# output_csv = "viirs_20_sao_1.csv"
# output_csv = "viirs_20_sao_2.csv"
# output_csv = "viirs_20_sao_3.csv"
# output_csv = "viirs_20_sao_4.csv"
# output_csv = "viirs_20_sao_5.csv"
# output_csv = "viirs_20_sao_6.csv"
# output_csv = "viirs_20_sao_7.csv"
# output_csv = "viirs_20_sao_8.csv"
# output_csv = "viirs_20_sao_9.csv"
# output_csv = "viirs_20_sao_10.csv"
# output_csv = "viirs_20_sao_11.csv"
output_csv ="viirs_20_sao_12.csv"

df = pd.read_csv(output_csv)
df.head()

,longitude,latitude,light_intensity
0,-53.133333,-19.808333,0.0
1,-53.129167,-19.808333,0.0
2,-53.125000,-19.808333,0.0
3,-53.120833,-19.808333,0.0
4,-53.116667,-19.808333,0.0
